This notebook identifies subnational units where the infection-to-incidence conversion and the incidence-to-mortality conversion are both amplified. The classification uses the same outcome-standardized WLS model contribution framework used in `figure2&3.ipynb` for the subnational units model contribution maps.

**Figure claim.** Dual-amplified subnational units mark places where climate exposure and informality jointly intensify both conversion steps: infection ecology to clinical incidence, then incidence to mortality.


In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.formula.api as smf
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerBase

warnings.filterwarnings("ignore", category=FutureWarning)

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "data" / "overall" / "admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp")

PROJECT_ROOT = find_project_root()
FINAL_SHP_PATH = PROJECT_ROOT / "data" / "overall" / "admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
FINAL_REGION_PATH = PROJECT_ROOT / "data" / "country" / "country_SSA_HE2.shp"
RESULT4_OUT_DIR = PROJECT_ROOT / "figure4"
RESULT4_OUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_HEAT_COL = "HE_perpop"
FINAL_FLOOD_COL = "sl_sevexp"
FINAL_EDI_COL = "EDI_qmean"
FINAL_FE_COL = "iso3"

mpl.rcParams.update({
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 7,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "axes.linewidth": 0.55,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.major.width": 0.55,
    "ytick.major.width": 0.55,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
})

COL_OTHER = "#F7F7F7"
COL_OTHER_EDGE = "white"
COL_LOW = "#D0D0D0"
COL_LOW_EDGE = "#8A8A8A"
COL_I2I = "#F0B864"
COL_I2M = "#A897C8"
COL_DUAL = "#E9827D"
COL_DUAL_EDGE = "#9E2F2C"
COL_DARK = "#2B2B2B"
COL_GRID = "#E7E7E7"

CLASS_ORDER = [
    "Low/other conversion",
    "Infection-to-incidence only",
    "Incidence-to-mortality only",
    "Dual amplification",
]
CLASS_COLORS = {
    "Low/other conversion": COL_LOW,
    "Infection-to-incidence only": COL_I2I,
    "Incidence-to-mortality only": COL_I2M,
    "Dual amplification": COL_DUAL,
}

# Plotting classes: explicit low is separated from middle/unclassified units.
VISIBLE_CLASS_ORDER = [
    "Dual amplification",
    "Low conversion",
    "Infection-to-incidence only",
    "Incidence-to-mortality only",
    "Other subnational units",
]
VISIBLE_CLASS_COLORS = {
    "Other subnational units": COL_OTHER,
    "Low conversion": COL_LOW,
    "Infection-to-incidence only": COL_I2I,
    "Incidence-to-mortality only": COL_I2M,
    "Dual amplification": COL_DUAL,
}


In [ ]:
MODEL_MAP_FORMULA = "log_ratio_z ~ heat_z + flood_z + heat_flood_z + edi_z + logpop_z"
CONTRIBUTION_COLS = [
    "map_edi_contribution_sd",
    "map_heat_contribution_sd",
    "map_flood_contribution_sd",
    "map_joint_contribution_sd",
]

def zscore(x):
    x = pd.to_numeric(x, errors="coerce")
    std = x.std(ddof=0)
    if not np.isfinite(std) or std == 0:
        return x * np.nan
    return (x - x.mean()) / std

def fit_cluster(formula, data, weight_col):
    return smf.wls(formula, data=data, weights=data[weight_col]).fit(
        cov_type="cluster", cov_kwds={"groups": data[FINAL_FE_COL]}
    )

def prepare_ratio(gdf, numerator_col, denominator_col, conversion):
    # Same ratio construction as 2_2.ipynb final_prepare_ratio.
    d = gdf.copy()
    d["orig_index"] = d.index
    d["ratio_numer"] = pd.to_numeric(d[numerator_col], errors="coerce")
    d["ratio_denom"] = pd.to_numeric(d[denominator_col], errors="coerce")
    d["ratio_weight"] = d["ratio_denom"]
    d["conversion_ratio"] = d["ratio_numer"] / d["ratio_denom"]
    invalid = (d["ratio_denom"] <= 0) | (d["ratio_numer"] < 0) | (d["conversion_ratio"] < 0)
    d.loc[invalid, ["conversion_ratio", "ratio_weight"]] = np.nan
    d["log_ratio"] = np.log1p(d["conversion_ratio"])
    d["heat_z"] = zscore(d[FINAL_HEAT_COL])
    d["flood_z"] = zscore(d[FINAL_FLOOD_COL])
    d["edi_z"] = zscore(d[FINAL_EDI_COL])
    d["logpop_z"] = zscore(np.log1p(d["ratio_weight"]))
    d["high_heat75"] = (d[FINAL_HEAT_COL] >= d[FINAL_HEAT_COL].quantile(0.75)).astype(int)
    d["high_flood75"] = (d[FINAL_FLOOD_COL] >= d[FINAL_FLOOD_COL].quantile(0.75)).astype(int)
    d["joint_heat_flood75"] = ((d["high_heat75"] == 1) & (d["high_flood75"] == 1)).astype(int)
    d["conversion"] = conversion
    needed = ["log_ratio", "conversion_ratio", "ratio_weight", "heat_z", "flood_z", "edi_z", "logpop_z", FINAL_FE_COL]
    return d.dropna(subset=needed).copy()

def model_component_sd(model, df, terms):
    # Same component decomposition as 2_2.ipynb model contribution maps.
    comp = np.zeros(len(df), dtype=float)
    for term in terms:
        if term in model.params.index and term in df.columns:
            comp += float(model.params[term]) * pd.to_numeric(df[term], errors="coerce").fillna(0).to_numpy()
    return comp

def add_model_map_fields(source_df, conversion_label):
    # Exact continuous contribution-map model from 2_2.ipynb:
    # outcome-standardized WLS, denominator weights, country-clustered SE.
    out = source_df.copy()
    out["log_ratio_z"] = zscore(out["log_ratio"])
    out["heat_flood_z"] = (
        pd.to_numeric(out["heat_z"], errors="coerce")
        * pd.to_numeric(out["flood_z"], errors="coerce")
    )
    model = fit_cluster(MODEL_MAP_FORMULA, out, "ratio_weight")
    out["map_pred_z"] = model.predict(out)
    out["map_residual_z"] = out["log_ratio_z"] - out["map_pred_z"]
    out["map_edi_contribution_sd"] = model_component_sd(model, out, ["edi_z"])
    out["map_heat_contribution_sd"] = model_component_sd(model, out, ["heat_z"])
    out["map_flood_contribution_sd"] = model_component_sd(model, out, ["flood_z"])
    out["map_joint_contribution_sd"] = model_component_sd(model, out, ["heat_flood_z"])
    out["mechanism_contribution_sd"] = out[CONTRIBUTION_COLS].sum(axis=1)
    out["map_conversion"] = conversion_label
    return out, model

def add_region(gdf, source_gdf):
    country_regions = gpd.read_file(FINAL_REGION_PATH)[["country_id", "country_na", "region"]].copy()
    country_regions["country_id4"] = country_regions["country_id"].astype(str).str.extract(r"(\d{4})", expand=False)
    country_lookup = (
        country_regions
        .dropna(subset=["country_id4"])
        .drop_duplicates("country_id4")
        .set_index("country_id4")
    )
    region_map = country_lookup["region"]
    country_name_map = country_lookup["country_na"]
    lookup = pd.DataFrame(index=source_gdf.index)
    lookup["country_id4"] = source_gdf["adm2ID"].astype(str).str.extract(r"(\d{4})", expand=False)
    lookup["region"] = lookup["country_id4"].map(region_map)
    lookup["country_name"] = lookup["country_id4"].map(country_name_map)
    out = gdf.copy()
    out["region"] = lookup.reindex(out["orig_index"])["region"].values
    out["country_name"] = lookup.reindex(out["orig_index"])["country_name"].values
    return out

final_gdf = gpd.read_file(FINAL_SHP_PATH)
final_gdf[FINAL_FE_COL] = final_gdf[FINAL_FE_COL].astype(str)

# Same numerator/denominator choices as 2_2.ipynb. The label below states the
# epidemiological direction represented by incidence / infection.
i2i = prepare_ratio(final_gdf, "sl_pfinc", "sl_pfinf", "infection_to_incidence")
i2m = prepare_ratio(final_gdf, "sl_pfmort", "sl_pfinc", "incidence_to_mortality")

i2i, i2i_model = add_model_map_fields(i2i, "infection_to_incidence")
i2m, i2m_model = add_model_map_fields(i2m, "incidence_to_mortality")
i2i = add_region(i2i, final_gdf)
i2m = add_region(i2m, final_gdf)

model_summary = pd.concat([
    pd.DataFrame({
        "conversion": "infection_to_incidence",
        "model_formula": MODEL_MAP_FORMULA,
        "term": i2i_model.params.index,
        "coef": i2i_model.params.values,
        "se": i2i_model.bse.values,
        "pval": i2i_model.pvalues.values,
        "n": int(i2i_model.nobs),
    }),
    pd.DataFrame({
        "conversion": "incidence_to_mortality",
        "model_formula": MODEL_MAP_FORMULA,
        "term": i2m_model.params.index,
        "coef": i2m_model.params.values,
        "se": i2m_model.bse.values,
        "pval": i2m_model.pvalues.values,
        "n": int(i2m_model.nobs),
    }),
], ignore_index=True)
model_summary.to_csv(RESULT4_OUT_DIR / "result4_wls_model_summary.csv", index=False, encoding="utf-8-sig")

print("Prepared conversion datasets with the 2_2.ipynb contribution-map model:")
print("infection-to-incidence n =", len(i2i))
print("incidence-to-mortality n =", len(i2m))
print(model_summary.to_string(index=False))


In [ ]:
def build_dual_amplification_table(i2i_df, i2m_df, threshold_q=0.75):
    keep_base = [
        "orig_index", "adm2ID", "iso3", "NAME_2", "region", "country_name", "sl_pop", FINAL_HEAT_COL, FINAL_FLOOD_COL, FINAL_EDI_COL,
        "geometry",
    ]
    i2i_keep = keep_base + [
        "conversion_ratio", "log_ratio", "log_ratio_z", "map_pred_z", "map_residual_z",
        "map_edi_contribution_sd", "map_heat_contribution_sd", "map_flood_contribution_sd", "map_joint_contribution_sd",
        "mechanism_contribution_sd", "ratio_weight",
    ]
    i2m_keep = [
        "orig_index", "conversion_ratio", "log_ratio", "log_ratio_z", "map_pred_z", "map_residual_z",
        "map_edi_contribution_sd", "map_heat_contribution_sd", "map_flood_contribution_sd", "map_joint_contribution_sd",
        "mechanism_contribution_sd", "ratio_weight",
    ]
    left = i2i_df[i2i_keep].rename(columns={
        "conversion_ratio": "i2i_ratio",
        "log_ratio": "i2i_log_ratio",
        "log_ratio_z": "i2i_observed_z",
        "map_pred_z": "i2i_model_pred_z",
        "map_residual_z": "i2i_model_residual_z",
        "map_edi_contribution_sd": "i2i_edi_contribution_sd",
        "map_heat_contribution_sd": "i2i_heat_contribution_sd",
        "map_flood_contribution_sd": "i2i_flood_contribution_sd",
        "map_joint_contribution_sd": "i2i_joint_contribution_sd",
        "mechanism_contribution_sd": "i2i_mechanism_contribution_sd",
        "ratio_weight": "i2i_weight",
    })
    right = i2m_df[i2m_keep].rename(columns={
        "conversion_ratio": "i2m_ratio",
        "log_ratio": "i2m_log_ratio",
        "log_ratio_z": "i2m_observed_z",
        "map_pred_z": "i2m_model_pred_z",
        "map_residual_z": "i2m_model_residual_z",
        "map_edi_contribution_sd": "i2m_edi_contribution_sd",
        "map_heat_contribution_sd": "i2m_heat_contribution_sd",
        "map_flood_contribution_sd": "i2m_flood_contribution_sd",
        "map_joint_contribution_sd": "i2m_joint_contribution_sd",
        "mechanism_contribution_sd": "i2m_mechanism_contribution_sd",
        "ratio_weight": "i2m_weight",
    })
    out = left.merge(right, on="orig_index", how="inner")
    out = gpd.GeoDataFrame(out, geometry="geometry", crs=i2i_df.crs)

    i2i_cut = float(out["i2i_mechanism_contribution_sd"].quantile(threshold_q))
    i2m_cut = float(out["i2m_mechanism_contribution_sd"].quantile(threshold_q))
    low_q = 1 - threshold_q
    i2i_low_cut = float(out["i2i_mechanism_contribution_sd"].quantile(low_q))
    i2m_low_cut = float(out["i2m_mechanism_contribution_sd"].quantile(low_q))
    out["i2i_high_amp"] = out["i2i_mechanism_contribution_sd"] >= i2i_cut
    out["i2m_high_amp"] = out["i2m_mechanism_contribution_sd"] >= i2m_cut
    out["i2i_low_conversion"] = out["i2i_mechanism_contribution_sd"] <= i2i_low_cut
    out["i2m_low_conversion"] = out["i2m_mechanism_contribution_sd"] <= i2m_low_cut
    out["clear_low_conversion"] = out["i2i_low_conversion"] & out["i2m_low_conversion"]
    out["dual_class"] = "Low/other conversion"
    out.loc[out["i2i_high_amp"] & ~out["i2m_high_amp"], "dual_class"] = "Infection-to-incidence only"
    out.loc[~out["i2i_high_amp"] & out["i2m_high_amp"], "dual_class"] = "Incidence-to-mortality only"
    out.loc[out["i2i_high_amp"] & out["i2m_high_amp"], "dual_class"] = "Dual amplification"

    obs_i2i_cut = float(out["i2i_observed_z"].quantile(threshold_q))
    obs_i2m_cut = float(out["i2m_observed_z"].quantile(threshold_q))
    out["observed_dual_high"] = (out["i2i_observed_z"] >= obs_i2i_cut) & (out["i2m_observed_z"] >= obs_i2m_cut)

    meta = {
        "threshold_quantile": threshold_q,
        "i2i_contribution_cut_sd": i2i_cut,
        "i2m_contribution_cut_sd": i2m_cut,
        "i2i_low_conversion_cut_sd": i2i_low_cut,
        "i2m_low_conversion_cut_sd": i2m_low_cut,
        "i2i_observed_cut_z": obs_i2i_cut,
        "i2m_observed_cut_z": obs_i2m_cut,
        "model_formula": MODEL_MAP_FORMULA,
        "model_match": "Same as 2_2.ipynb continuous model contribution maps for subnational units",
    }
    return out, meta

def summarize_regions(dual_gdf):
    d = dual_gdf.copy()
    d["pop_weight"] = pd.to_numeric(d["sl_pop"], errors="coerce").clip(lower=0)
    rows = []
    for region, reg in d.dropna(subset=["region"]).groupby("region"):
        total_n = len(reg)
        total_pop = reg["pop_weight"].sum()
        for cls in CLASS_ORDER:
            sub = reg[reg["dual_class"] == cls]
            pop = sub["pop_weight"].sum()
            rows.append({
                "region": region,
                "dual_class": cls,
                "admin2_n": int(len(sub)),
                "admin2_total": int(total_n),
                "admin2_share": len(sub) / total_n if total_n else np.nan,
                "population": float(pop),
                "population_total": float(total_pop),
                "population_share": pop / total_pop if total_pop > 0 else np.nan,
            })
    return pd.DataFrame(rows)

def summarize_countries(dual_gdf):
    d = dual_gdf.copy()
    d["pop_weight"] = pd.to_numeric(d["sl_pop"], errors="coerce").clip(lower=0)
    d["dual_score"] = d["i2i_mechanism_contribution_sd"] + d["i2m_mechanism_contribution_sd"]
    rows = []
    for (iso3, country_name), sub in d.dropna(subset=["iso3"]).groupby(["iso3", "country_name"], dropna=False):
        dual = sub[sub["dual_class"] == "Dual amplification"].copy()
        i2i_only = sub[sub["dual_class"] == "Infection-to-incidence only"].copy()
        i2m_only = sub[sub["dual_class"] == "Incidence-to-mortality only"].copy()
        low_other = sub[sub["dual_class"] == "Low/other conversion"].copy()
        total_n = len(sub)
        total_pop = sub["pop_weight"].sum()
        dual_pop = dual["pop_weight"].sum()
        i2i_pop = i2i_only["pop_weight"].sum()
        i2m_pop = i2m_only["pop_weight"].sum()
        rows.append({
            "iso3": iso3,
            "country_name": country_name if pd.notna(country_name) else iso3,
            "admin2_total": int(total_n),
            "low_other_admin2_n": int(len(low_other)),
            "i2i_only_admin2_n": int(len(i2i_only)),
            "i2m_only_admin2_n": int(len(i2m_only)),
            "dual_admin2_n": int(len(dual)),
            "i2i_only_admin2_share": len(i2i_only) / total_n if total_n else np.nan,
            "i2m_only_admin2_share": len(i2m_only) / total_n if total_n else np.nan,
            "dual_admin2_share": len(dual) / total_n if total_n else np.nan,
            "i2i_or_dual_admin2_share": (len(i2i_only) + len(dual)) / total_n if total_n else np.nan,
            "i2m_or_dual_admin2_share": (len(i2m_only) + len(dual)) / total_n if total_n else np.nan,
            "i2i_only_population": float(i2i_pop),
            "i2m_only_population": float(i2m_pop),
            "dual_population": float(dual_pop),
            "population_total": float(total_pop),
            "dual_population_share": dual_pop / total_pop if total_pop > 0 else np.nan,
            "mean_i2i_contribution_sd": float(dual["i2i_mechanism_contribution_sd"].mean()) if len(dual) else np.nan,
            "mean_i2m_contribution_sd": float(dual["i2m_mechanism_contribution_sd"].mean()) if len(dual) else np.nan,
            "mean_dual_score_sd": float(dual["dual_score"].mean()) if len(dual) else np.nan,
        })
    return pd.DataFrame(rows).sort_values(["dual_admin2_n", "dual_admin2_share", "iso3"], ascending=[False, False, True])

dual_gdf, threshold_meta = build_dual_amplification_table(i2i, i2m, threshold_q=0.75)
region_summary = summarize_regions(dual_gdf)
country_summary = summarize_countries(dual_gdf)

source_no_geom = dual_gdf.drop(columns="geometry", errors="ignore")
source_no_geom.to_csv(RESULT4_OUT_DIR / "result4_dual_amplification_source_data.csv", index=False, encoding="utf-8-sig")
region_summary.to_csv(RESULT4_OUT_DIR / "result4_dual_amplification_region_summary.csv", index=False, encoding="utf-8-sig")
country_summary.to_csv(RESULT4_OUT_DIR / "result4_dual_amplification_country_summary.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([threshold_meta]).to_csv(RESULT4_OUT_DIR / "result4_dual_amplification_thresholds.csv", index=False, encoding="utf-8-sig")

print("Thresholds:", threshold_meta)
print("Class counts:")
print(dual_gdf["dual_class"].value_counts().reindex(CLASS_ORDER).fillna(0).astype(int).to_string())
print("Observed dual overlap among model-defined dual admin2:", int(dual_gdf.loc[dual_gdf["dual_class"] == "Dual amplification", "observed_dual_high"].sum()))


In [ ]:
def get_country_boundaries():
    countries = gpd.read_file(FINAL_REGION_PATH)[["geometry"]].dropna(subset=["geometry"]).to_crs("EPSG:4326")
    return countries.boundary, countries.dissolve().boundary

def save_pub_figure(fig, out_base, dpi=600):
    fig.savefig(str(out_base) + ".pdf", bbox_inches="tight")

def scale_area(values, min_area=22, max_area=760):
    values = np.asarray(values, dtype=float)
    if values.size == 0 or np.nanmax(values) <= np.nanmin(values):
        return np.full_like(values, (min_area + max_area) / 2, dtype=float)
    scaled = (values - np.nanmin(values)) / (np.nanmax(values) - np.nanmin(values))
    return min_area + scaled * (max_area - min_area)

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(length=2.5, width=0.55, color="0.25")


## figure4a&b

In [ ]:
def _bubble_size_from_dual_score(values, reference_values=None):
    values = pd.to_numeric(pd.Series(values), errors="coerce").fillna(0)
    ref = values if reference_values is None else pd.to_numeric(pd.Series(reference_values), errors="coerce").fillna(0)
    lo = float(ref.quantile(0.05)) if len(ref) else 0.0
    hi = float(ref.quantile(0.95)) if len(ref) else 1.0
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = float(ref.min()) if len(ref) else 0.0
        hi = float(ref.max()) if len(ref) else 1.0
    if hi <= lo:
        hi = lo + 1e-6
    scaled = ((values.clip(lo, hi) - lo) / (hi - lo)).pow(0.75)
    return 18 + 230 * scaled

def _dual_score_values(gdf):
    dual_values = gdf[gdf["dual_class"] == "Dual amplification"].copy()
    if dual_values.empty:
        return pd.Series(dtype=float)
    values = dual_values["i2i_mechanism_contribution_sd"] + dual_values["i2m_mechanism_contribution_sd"]
    return pd.to_numeric(values, errors="coerce").dropna()

def _dual_score_legend_values(values):
    values = pd.Series(values, dtype=float).dropna()
    if values.empty:
        return np.array([0.0, 0.5, 1.0])
    score_values = values.quantile([0.25, 0.50, 0.75]).to_numpy()
    score_values = np.unique(np.round(score_values, 2))
    if len(score_values) < 3:
        lo = float(values.min())
        hi = float(values.max())
        if hi <= lo:
            hi = lo + 1e-6
        score_values = np.linspace(lo, hi, 3)
    return score_values

def _light_to_dark_cmap(name, dark_color):
    return mpl.colors.LinearSegmentedColormap.from_list(name, ["#F7F7F7", dark_color])

def _robust_norm(values):
    values = pd.to_numeric(values, errors="coerce").dropna()
    if values.empty:
        return mpl.colors.Normalize(vmin=0, vmax=1)
    lo = float(values.quantile(0.05))
    hi = float(values.quantile(0.95))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = float(values.min())
        hi = float(values.max())
    if hi <= lo:
        hi = lo + 1e-6
    return mpl.colors.Normalize(vmin=lo, vmax=hi)

class _GradientLegendHandle:
    def __init__(self, cmap):
        self.cmap = cmap

class _HandlerGradient(HandlerBase):
    def __init__(self, n_stripes=14, **kwargs):
        super().__init__(**kwargs)
        self.n_stripes = n_stripes

    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        artists = []
        stripe_width = width / self.n_stripes
        for j in range(self.n_stripes):
            artists.append(
                mpl.patches.Rectangle(
                    (xdescent + j * stripe_width, ydescent),
                    stripe_width + 0.02,
                    height,
                    transform=trans,
                    facecolor=orig_handle.cmap((j + 0.5) / self.n_stripes),
                    edgecolor="none",
                    linewidth=0,
                )
            )
        artists.append(
            mpl.patches.Rectangle(
                (xdescent, ydescent), width, height,
                transform=trans, facecolor="none", edgecolor="white", linewidth=0.35,
            )
        )
        return artists

def _shared_class_handles(marker_size=5.8):
    return [
        Patch(facecolor=COL_DUAL, edgecolor=COL_DUAL_EDGE, linewidth=1.15, label="Dual amplification"),
        Patch(facecolor=COL_LOW, edgecolor=COL_LOW_EDGE, linewidth=0.95, label="Low conversion"),
        _GradientLegendHandle(_light_to_dark_cmap("legend_i2i_single_amp", COL_I2I)),
        _GradientLegendHandle(_light_to_dark_cmap("legend_i2m_single_amp", COL_I2M)),
        Patch(facecolor=COL_OTHER, edgecolor=COL_OTHER_EDGE, linewidth=0.85, label="Other subnational units"),
    ]

def _shared_size_handles(reference_values, edgecolor="0.25", linewidth=0.60):
    score_values = _dual_score_legend_values(reference_values)
    score_sizes = _bubble_size_from_dual_score(score_values, reference_values)
    handles = [plt.scatter([], [], s=s, facecolor="none", edgecolor=edgecolor, linewidth=linewidth) for s in score_sizes]
    labels = [f"{v:.2f}" for v in score_values]
    return handles, labels

def _add_shared_legends(fig, dual_reference_values=None):
    fig.legend(
        handles=_shared_class_handles(marker_size=5.4),
        labels=[
            "Dual amplification",
            "Low conversion",
            "Infection-to-incidence only",
            "Incidence-to-mortality only",
            "Other subnational units",
        ],
        handler_map={_GradientLegendHandle: _HandlerGradient()},
        loc="lower left",
        bbox_to_anchor=(0.035, 0.160), ncol=1, frameon=False,
        fontsize=6, title="Subnational-unit class", title_fontsize=6,
        handlelength=1.45, handletextpad=0.45, labelspacing=0.34,
    )

def plot_single_maps_dual_bubbles(ax, dual_gdf, add_legends=True):
    country_gdf = gpd.read_file(FINAL_REGION_PATH)[["region", "geometry"]].dropna(subset=["geometry"]).to_crs("EPSG:4326")
    region_boundaries = country_gdf.dissolve(by="region").boundary
    plot_gdf = dual_gdf.to_crs("EPSG:4326") if dual_gdf.crs is not None else dual_gdf.copy()
    plot_gdf = plot_gdf.copy()
    plot_gdf["point_geom"] = plot_gdf.geometry.representative_point()
    plot_gdf["x"] = plot_gdf["point_geom"].x
    plot_gdf["y"] = plot_gdf["point_geom"].y

    class_mask = (
        plot_gdf["clear_low_conversion"]
        | plot_gdf["dual_class"].isin(["Infection-to-incidence only", "Incidence-to-mortality only", "Dual amplification"])
    )
    other_units = plot_gdf[~class_mask].copy()
    low_conversion = plot_gdf[plot_gdf["clear_low_conversion"]].copy()
    i2i_only = plot_gdf[plot_gdf["dual_class"] == "Infection-to-incidence only"].copy()
    i2m_only = plot_gdf[plot_gdf["dual_class"] == "Incidence-to-mortality only"].copy()
    dual = plot_gdf[plot_gdf["dual_class"] == "Dual amplification"].copy()
    dual_reference_values = _dual_score_values(plot_gdf)

    country_gdf.plot(ax=ax, color=COL_OTHER, edgecolor="white", linewidth=0.25, zorder=0)
    if len(other_units):
        other_units.plot(ax=ax, color=COL_OTHER, edgecolor=COL_OTHER_EDGE, linewidth=0.08, alpha=0.98, zorder=0.5)
    if len(low_conversion):
        low_conversion.plot(ax=ax, color=COL_LOW, edgecolor=COL_LOW_EDGE, linewidth=0.26, alpha=0.96, zorder=1)
    i2i_cmap = _light_to_dark_cmap("i2i_single_amp", COL_I2I)
    i2m_cmap = _light_to_dark_cmap("i2m_single_amp", COL_I2M)
    i2i_norm = _robust_norm(i2i_only["i2i_mechanism_contribution_sd"])
    i2m_norm = _robust_norm(i2m_only["i2m_mechanism_contribution_sd"])

    if len(i2i_only):
        i2i_only["plot_intensity"] = i2i_only["i2i_mechanism_contribution_sd"].clip(i2i_norm.vmin, i2i_norm.vmax)
        i2i_only.plot(ax=ax, column="plot_intensity", cmap=i2i_cmap, norm=i2i_norm, edgecolor="white", linewidth=0.10, alpha=0.96, zorder=2)
    if len(i2m_only):
        i2m_only["plot_intensity"] = i2m_only["i2m_mechanism_contribution_sd"].clip(i2m_norm.vmin, i2m_norm.vmax)
        i2m_only.plot(ax=ax, column="plot_intensity", cmap=i2m_cmap, norm=i2m_norm, edgecolor="white", linewidth=0.10, alpha=0.96, zorder=3)

    country_gdf.boundary.plot(ax=ax, color="0.5", linewidth=0.20, zorder=4)
    country_gdf.dissolve().boundary.plot(ax=ax, color="0.22", linewidth=0.62, zorder=5)
    region_boundaries.plot(ax=ax, color="0.5", linewidth=0.72, zorder=6)
    if len(dual):
        dual.plot(ax=ax, color=COL_DUAL, edgecolor=COL_DUAL_EDGE, linewidth=0.34, alpha=0.88, zorder=7)

    ax.set_xlim(-20, 55)
    ax.set_ylim(-37, 29)
    ax.set_aspect("equal")
    ax.set_axis_off()

    if add_legends:
        _add_shared_legends(ax.figure, dual_reference_values)
    return dual_reference_values

def _selected_zoom_countries(country_summary, top_n=6):
    return (
        country_summary[country_summary["dual_admin2_n"] > 0]
        .sort_values(["dual_admin2_share", "dual_admin2_n"], ascending=False)
        .head(top_n)
        .copy()
    )

def _plot_selected_country_boundaries_on_main(ax, country_summary, top_n=9):
    selected = _selected_zoom_countries(country_summary, top_n=top_n)
    selected_names = set(selected["country_name"].dropna().astype(str))
    country_outline_gdf = gpd.read_file(FINAL_REGION_PATH)[["country_na", "geometry"]].dropna(subset=["geometry"]).to_crs("EPSG:4326")
    highlight = country_outline_gdf[country_outline_gdf["country_na"].astype(str).isin(selected_names)].copy()
    if len(highlight):
        highlight.boundary.plot(ax=ax, color="0.3", linewidth=0.95, zorder=8)
    return highlight

def _degree_tick_values(vmin, vmax, target=3):
    span = float(vmax - vmin)
    if not np.isfinite(span) or span <= 0:
        return []
    steps = np.array([0.25, 0.5, 1, 2, 5, 10, 20, 30], dtype=float)
    ticks = np.array([], dtype=float)
    for step in steps:
        start = np.ceil(vmin / step) * step
        stop = np.floor(vmax / step) * step
        candidate = np.arange(start, stop + step * 0.25, step)
        if len(candidate) >= 2:
            ticks = candidate
            if len(candidate) <= target:
                break
    if len(ticks) < 2:
        step = steps[0]
        ticks = np.array([np.ceil(vmin / step) * step, np.floor(vmax / step) * step])
    ticks = np.unique(np.round(ticks, 2))
    if len(ticks) > target:
        idx = np.linspace(0, len(ticks) - 1, target).round().astype(int)
        ticks = ticks[idx]
    return ticks

def _format_lon_tick(value, pos=None):
    hemi = "E" if value >= 0 else "W"
    value = abs(float(value))
    return f"{value:.0f}{hemi}" if abs(value - round(value)) < 0.05 else f"{value:.1f}{hemi}"

def _format_lat_tick(value, pos=None):
    hemi = "N" if value >= 0 else "S"
    value = abs(float(value))
    return f"{value:.0f}{hemi}" if abs(value - round(value)) < 0.05 else f"{value:.1f}{hemi}"

def _style_lonlat_frame(ax, xmin, xmax, ymin, ymax):
    ax.set_axis_on()
    ax.set_axisbelow(True)
    ax.set_xticks(_degree_tick_values(xmin, xmax, target=3))
    ax.set_yticks(_degree_tick_values(ymin, ymax, target=3))
    ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(_format_lon_tick))
    ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(_format_lat_tick))
    ax.tick_params(
        axis="both", which="major", direction="out",
        length=1.8, width=0.35, color="0.48",
        labelsize=4.4, labelcolor="0.28", pad=1.0,
        top=False, right=False, labeltop=False, labelright=False,
    )
    ax.grid(True, color="0.86", linewidth=0.24, linestyle=(0, (1.2, 2.0)))
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("0.52")
        spine.set_linewidth(0.38)

def _plot_country_zoom_axes(axes, dual_gdf, country_summary, top_n=6):
    selected = _selected_zoom_countries(country_summary, top_n=top_n)
    selected_iso3 = selected["iso3"].tolist()
    selected_names = dict(zip(selected["iso3"], selected["country_name"]))

    full_gdf = dual_gdf.to_crs("EPSG:4326") if dual_gdf.crs is not None else dual_gdf.copy()
    dual_reference_values = _dual_score_values(full_gdf)
    plot_gdf = full_gdf[full_gdf[FINAL_FE_COL].isin(selected_iso3)].copy()
    plot_gdf["point_geom"] = plot_gdf.geometry.representative_point()
    plot_gdf["x"] = plot_gdf["point_geom"].x
    plot_gdf["y"] = plot_gdf["point_geom"].y
    country_outline_gdf = gpd.read_file(FINAL_REGION_PATH)[["country_na", "geometry"]].dropna(subset=["geometry"]).to_crs("EPSG:4326")

    i2i_norm = _robust_norm(plot_gdf.loc[plot_gdf["dual_class"] == "Infection-to-incidence only", "i2i_mechanism_contribution_sd"])
    i2m_norm = _robust_norm(plot_gdf.loc[plot_gdf["dual_class"] == "Incidence-to-mortality only", "i2m_mechanism_contribution_sd"])
    i2i_cmap = _light_to_dark_cmap("zoom_i2i_single_amp", COL_I2I)
    i2m_cmap = _light_to_dark_cmap("zoom_i2m_single_amp", COL_I2M)

    axes = np.asarray(axes).ravel()
    for ax, iso3 in zip(axes, selected_iso3):
        sub = plot_gdf[plot_gdf[FINAL_FE_COL] == iso3].copy()
        class_mask = (
            sub["clear_low_conversion"]
            | sub["dual_class"].isin(["Infection-to-incidence only", "Incidence-to-mortality only", "Dual amplification"])
        )
        other_units = sub[~class_mask].copy()
        low_conversion = sub[sub["clear_low_conversion"]].copy()
        dual = sub[sub["dual_class"] == "Dual amplification"].copy()
        i2i_only = sub[sub["dual_class"] == "Infection-to-incidence only"].copy()
        i2m_only = sub[sub["dual_class"] == "Incidence-to-mortality only"].copy()

        if len(other_units):
            other_units.plot(ax=ax, color=COL_OTHER, edgecolor=COL_OTHER_EDGE, linewidth=0.12, alpha=0.98, zorder=0.5)
        if len(low_conversion):
            low_conversion.plot(ax=ax, color=COL_LOW, edgecolor=COL_LOW_EDGE, linewidth=0.36, alpha=0.96, zorder=1)
        if len(i2i_only):
            i2i_only["plot_intensity"] = i2i_only["i2i_mechanism_contribution_sd"].clip(i2i_norm.vmin, i2i_norm.vmax)
            i2i_only.plot(ax=ax, column="plot_intensity", cmap=i2i_cmap, norm=i2i_norm, edgecolor="white", linewidth=0.14, alpha=0.96, zorder=2)
        if len(i2m_only):
            i2m_only["plot_intensity"] = i2m_only["i2m_mechanism_contribution_sd"].clip(i2m_norm.vmin, i2m_norm.vmax)
            i2m_only.plot(ax=ax, column="plot_intensity", cmap=i2m_cmap, norm=i2m_norm, edgecolor="white", linewidth=0.14, alpha=0.96, zorder=3)
        if len(dual):
            dual.plot(ax=ax, color=COL_DUAL, edgecolor=COL_DUAL_EDGE, linewidth=0.45, alpha=0.88, zorder=5)

        country_name = selected_names.get(iso3, iso3)
        outline = country_outline_gdf[country_outline_gdf["country_na"] == country_name]
        if len(outline):
            outline.boundary.plot(ax=ax, color="0.3", linewidth=0.62, zorder=6)
        else:
            sub.boundary.plot(ax=ax, color="0.3", linewidth=0.28, zorder=6)
        xmin, ymin, xmax, ymax = sub.total_bounds
        dx = max((xmax - xmin) * 0.06, 0.25)
        dy = max((ymax - ymin) * 0.06, 0.25)
        x0, x1 = xmin - dx, xmax + dx
        y0, y1 = ymin - dy, ymax + dy
        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)
        ax.set_aspect("equal")
        _style_lonlat_frame(ax, x0, x1, y0, y1)
        ax.set_title(str(selected_names.get(iso3, iso3)), loc="center", fontsize=6.8, pad=1.8)

    for ax in axes[len(selected_iso3):]:
        ax.set_visible(False)

    zoom_source = selected[["iso3", "country_name", "dual_admin2_share", "dual_admin2_n", "admin2_total"]].copy()
    zoom_source = zoom_source.rename(columns={
        "dual_admin2_share": "dual_subnational_unit_share",
        "dual_admin2_n": "dual_subnational_unit_n",
        "admin2_total": "subnational_unit_total",
    })
    zoom_source.to_csv(RESULT4_OUT_DIR / "result4_extra_country_zoom_maps_source_data.csv", index=False)
    return selected, dual_reference_values

def plot_country_zoom_maps(dual_gdf, country_summary, top_n=9, add_legends=True):
    ncols = 3
    nrows = int(np.ceil(top_n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.9, 2.05 * nrows))
    _, dual_reference_values = _plot_country_zoom_axes(axes, dual_gdf, country_summary, top_n=top_n)
    if add_legends:
        _add_shared_legends(fig, dual_reference_values)
    fig.subplots_adjust(left=0.03, right=0.99, top=0.93, bottom=0.12, wspace=0.03, hspace=0.12)
    return fig

def plot_figure4a_composite(dual_gdf, country_summary, top_n=9):
    fig = plt.figure(figsize=(8.6, 4.9))
    gs = fig.add_gridspec(
        nrows=3, ncols=5,
        width_ratios=[1.45, 1.45, 0.68, 0.68, 0.68],
        left=0.01, right=0.99, top=0.965, bottom=0.16,
        wspace=0.025, hspace=0.12,
    )
    main_ax = fig.add_subplot(gs[:, :2])
    zoom_axes = [fig.add_subplot(gs[r, c]) for r in range(3) for c in range(2, 5)]
    dual_reference_values = plot_single_maps_dual_bubbles(main_ax, dual_gdf, add_legends=False)
    _plot_selected_country_boundaries_on_main(main_ax, country_summary, top_n=top_n)
    _plot_country_zoom_axes(zoom_axes, dual_gdf, country_summary, top_n=top_n)
    _add_shared_legends(fig, dual_reference_values)
    return fig

fig = plot_figure4a_composite(dual_gdf, country_summary, top_n=9)
out_base = RESULT4_OUT_DIR / "figure4a&b"
save_pub_figure(fig, out_base)
plt.show()


## figure4a_2

In [ ]:
def _visible_class_subsets(d):
    class_mask = (
        d["clear_low_conversion"]
        | d["dual_class"].isin(["Infection-to-incidence only", "Incidence-to-mortality only", "Dual amplification"])
    )
    return [
        ("Other subnational units", d[~class_mask].copy()),
        ("Low conversion", d[d["clear_low_conversion"]].copy()),
        ("Infection-to-incidence only", d[d["dual_class"] == "Infection-to-incidence only"].copy()),
        ("Incidence-to-mortality only", d[d["dual_class"] == "Incidence-to-mortality only"].copy()),
        ("Dual amplification", d[d["dual_class"] == "Dual amplification"].copy()),
    ]

def plot_quadrant(ax, dual_gdf, threshold_meta):
    d = dual_gdf.copy()
    for cls, sub in _visible_class_subsets(d):
        if sub.empty:
            continue
        size = 5 if cls == "Other subnational units" else (9 if cls != "Dual amplification" else 15)
        alpha = 0.22 if cls == "Other subnational units" else (0.58 if cls == "Low conversion" else 0.72)
        ax.scatter(
            sub["i2i_mechanism_contribution_sd"],
            sub["i2m_mechanism_contribution_sd"],
            s=size,
            color=VISIBLE_CLASS_COLORS[cls],
            alpha=alpha,
            linewidths=0,
            rasterized=True,
        )
    ax.axvline(threshold_meta["i2i_contribution_cut_sd"], color="0.25", lw=0.75, ls=(0, (2, 2)))
    ax.axhline(threshold_meta["i2m_contribution_cut_sd"], color="0.25", lw=0.75, ls=(0, (2, 2)))

    ax.set_xlabel("Infection-to-incidence contribution (SD)", fontsize=10)
    ax.set_ylabel("Incidence-to-mortality contribution (SD)", fontsize=10)
    ax.tick_params(axis="both", labelsize=9)
    ax.grid(color=COL_GRID, lw=0.45, zorder=0)

fig, ax = plt.subplots(figsize=(3.4, 3.4))
plot_quadrant(ax, dual_gdf, threshold_meta)
fig.subplots_adjust(left=0.13, right=0.98, top=0.90, bottom=0.16)
out_base = RESULT4_OUT_DIR / "figure4a_2"
save_pub_figure(fig, out_base)
plt.show()


## figure4c

In [ ]:
def _nice_population_legend_levels(max_value):
    candidates = np.array([100_000, 250_000, 500_000, 1_000_000, 2_500_000, 5_000_000, 10_000_000, 25_000_000], dtype=float)
    levels = candidates[candidates <= max_value]
    if len(levels) >= 3:
        return levels[-3:]
    if len(levels) > 0:
        return levels
    return np.array([max(max_value, 1.0)], dtype=float)


def plot_country_dual_summary_population_bubbles(ax, country_summary, top_n=10):
    d = country_summary.copy()
    d = d[d["admin2_total"] >= 5].copy()
    d["x_i2i_only_pct"] = d["i2i_only_admin2_share"] * 100
    d["y_i2m_only_pct"] = d["i2m_only_admin2_share"] * 100
    d["dual_unit_share_pct"] = d["dual_admin2_share"] * 100
    d["dual_pop_share_pct"] = d["dual_population_share"] * 100
    d["dual_population_million"] = d["dual_population"] / 1_000_000
    d = d.dropna(subset=["x_i2i_only_pct", "y_i2m_only_pct", "dual_population", "dual_pop_share_pct"]).copy()

    source_cols = [
        "iso3", "country_name", "admin2_total", "i2i_only_admin2_share", "i2m_only_admin2_share",
        "dual_admin2_n", "dual_admin2_share", "dual_population", "dual_population_share",
        "mean_dual_score_sd", "x_i2i_only_pct", "y_i2m_only_pct", "dual_pop_share_pct",
        "dual_population_million",
    ]
    d[source_cols].sort_values("dual_population", ascending=False).to_csv(
        RESULT4_OUT_DIR / "result4_panel_c_country_dual_burden_population_bubbles_source_data.csv",
        index=False,
        encoding="utf-8-sig",
    )

    vmax = max(float(d["dual_pop_share_pct"].quantile(0.98)), 1.0)
    norm = mpl.colors.Normalize(vmin=0, vmax=vmax)
    cmap = mpl.colormaps.get_cmap("plasma_r")
    sizes = scale_area(d["dual_population"].to_numpy(dtype=float), min_area=18, max_area=620)

    ax.set_facecolor("white")
    sc = ax.scatter(
        d["x_i2i_only_pct"],
        d["y_i2m_only_pct"],
        s=sizes,
        c=d["dual_pop_share_pct"],
        cmap=cmap,
        norm=norm,
        alpha=0.82,
        edgecolor="white",
        linewidth=0.45,
        zorder=3,
    )
    ax.grid(color="#E8E8E8", lw=0.7, zorder=0)

    label_df = d.sort_values(["dual_population", "dual_admin2_share"], ascending=False).head(top_n).copy()
    label_offsets = {
        "Nigeria": (20, 0),
        "Ghana": (8, 8),
        "Senegal": (8, 16),
        "Burkina Faso": (18, -2),
        "Benin": (8, -18),
        "Togo": (8, 3),
        "The Gambia": (8, -9),
        "Guinea-Bissau": (8, 9),
        "Eritrea": (8, -5),
        "Chad": (8, -5),
    }
    label_sizes = scale_area(label_df["dual_population"].to_numpy(dtype=float), min_area=18, max_area=620)
    ax.scatter(
        label_df["x_i2i_only_pct"],
        label_df["y_i2m_only_pct"],
        s=label_sizes,
        c=label_df["dual_pop_share_pct"],
        cmap=cmap,
        norm=norm,
        alpha=0.88,
        edgecolor="0.10",
        linewidth=0.65,
        zorder=4,
    )
    for _, row in label_df.iterrows():
        label = row["country_name"] if pd.notna(row["country_name"]) else row["iso3"]
        label = str(label).replace("Democratic Republic of the Congo", "DR Congo").replace("Central African Republic", "Central African Rep.")
        dx, dy = label_offsets.get(label, (8, 8))
        ax.annotate(
            label,
            xy=(row["x_i2i_only_pct"], row["y_i2m_only_pct"]),
            xytext=(dx, dy),
            textcoords="offset points",
            ha="left",
            va="center",
            fontsize=5.5,
            color=COL_DARK,
            bbox=dict(boxstyle="round,pad=0.12", facecolor="white", edgecolor="none", alpha=0.74),
            arrowprops=dict(arrowstyle="-", color="0.18", lw=0.35, shrinkA=1.5, shrinkB=2.0),
            zorder=6,
        )

    ax.set_xlabel("Infection-to-incidence-only subnational-unit share (%)", fontsize=7)
    ax.set_ylabel("Incidence-to-mortality-only subnational-unit share (%)", fontsize=7)
    ax.tick_params(axis="both", labelsize=7, length=2.5, width=0.55, color="0.25")
    clean_axes(ax)

    cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.025)
    cbar.set_label("Dual-amplified slum population share (%)", fontsize=7)
    cbar.ax.tick_params(labelsize=5.5, length=2)
    cbar.outline.set_linewidth(0.35)

    legend_levels = _nice_population_legend_levels(float(d["dual_population"].max()))
    legend_sizes = scale_area(legend_levels, min_area=18, max_area=620)
    handles = [plt.scatter([], [], s=s, facecolor="none", edgecolor="0.25", linewidth=0.65) for s in legend_sizes]
    labels = [f"{v / 1_000_000:g}" for v in legend_levels]
    leg = ax.legend(
        handles,
        labels,
        title="Dual-amplified\nslum population (million)",
        loc="upper right",
        frameon=False,
        fontsize=5.3,
        title_fontsize=5.5,
        labelspacing=3.2,
        borderaxespad=0.2,
        handletextpad=2
    )
    leg._legend_box.sep = 5   

fig, ax = plt.subplots(figsize=(5.2, 3.75))
plot_country_dual_summary_population_bubbles(ax, country_summary, top_n=10)
fig.subplots_adjust(left=0.15, right=0.88, top=0.90, bottom=0.16)
out_base = RESULT4_OUT_DIR / "figure4c"
save_pub_figure(fig, out_base)
plt.show()


## figure4d

In [ ]:
def make_visible_region_population_summary(dual_gdf):
    d = dual_gdf.copy()
    d["sl_pop"] = pd.to_numeric(d["sl_pop"], errors="coerce").fillna(0)
    region_order = [r for r in ["Western", "Central", "Eastern", "Northern", "Southern"] if r in d["region"].dropna().unique()]
    if not region_order:
        region_order = sorted(d["region"].dropna().unique())

    rows = []
    for region in region_order:
        reg = d[d["region"] == region].copy()
        total_n = len(reg)
        total_pop = reg["sl_pop"].sum()
        if total_n == 0:
            continue

        low_mask = reg["clear_low_conversion"].fillna(False).astype(bool)
        highlighted_mask = low_mask | reg["dual_class"].isin([
            "Infection-to-incidence only",
            "Incidence-to-mortality only",
            "Dual amplification",
        ])
        class_masks = {
            "Dual amplification": reg["dual_class"] == "Dual amplification",
            "Low conversion": low_mask,
            "Infection-to-incidence only": reg["dual_class"] == "Infection-to-incidence only",
            "Incidence-to-mortality only": reg["dual_class"] == "Incidence-to-mortality only",
            "Other subnational units": ~highlighted_mask,
        }

        for cls in VISIBLE_CLASS_ORDER:
            mask = class_masks[cls]
            n = int(mask.sum())
            pop = reg.loc[mask, "sl_pop"].sum()
            rows.append({
                "region": region,
                "dual_class": cls,
                "admin2_n": n,
                "total_admin2_n": total_n,
                "admin2_share": n / total_n,
                "population": pop,
                "total_population": total_pop,
                "population_share": pop / total_pop if total_pop > 0 else np.nan,
            })
    return pd.DataFrame(rows)


def plot_region_population_stack(ax, dual_gdf):
    d = make_visible_region_population_summary(dual_gdf)
    d.to_csv(RESULT4_OUT_DIR / "result4_panel_d_regional_class_composition_population_share_source_data.csv", index=False)

    region_order = [r for r in ["Western", "Central", "Eastern", "Northern", "Southern"] if r in d["region"].dropna().unique()]
    if not region_order:
        region_order = sorted(d["region"].dropna().unique())
    pivot = (
        d.pivot(index="region", columns="dual_class", values="population_share")
        .reindex(region_order)
        .reindex(columns=VISIBLE_CLASS_ORDER)
        .fillna(0)
    )

    bottom = np.zeros(len(pivot))
    x = np.arange(len(pivot))
    for cls in VISIBLE_CLASS_ORDER:
        vals = pivot[cls].to_numpy() * 100
        ax.bar(x, vals, bottom=bottom, color=VISIBLE_CLASS_COLORS[cls], width=0.68, edgecolor="white", linewidth=0.35)
        for xi, val, y0 in zip(x, vals, bottom):
            if val < 2.5:
                continue
            ax.text(
                xi,
                y0 + val / 2,
                f"{val:.1f}",
                ha="center",
                va="center",
                fontsize=4.6,
                color=COL_DARK,
                clip_on=True,
            )
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=35, ha="right")
    ax.set_ylim(0, 104)
    ax.set_ylabel("Slum population share (%)")
    legend_handles = [Patch(facecolor=VISIBLE_CLASS_COLORS[cls], edgecolor="white", label=cls) for cls in VISIBLE_CLASS_ORDER]
    ax.legend(
        handles=legend_handles,
        loc="lower left",
        bbox_to_anchor=(0.0, 1.02),
        ncol=2,
        frameon=False,
        fontsize=5.3,
        title_fontsize=5.6,
        handlelength=1.2,
        columnspacing=0.9,
        borderaxespad=0,
    )


fig, ax = plt.subplots(figsize=(3.8, 3.6))
plot_region_population_stack(ax, dual_gdf)
fig.subplots_adjust(left=0.17, right=0.98, top=0.80, bottom=0.20)
out_base = RESULT4_OUT_DIR / "figure4d"
save_pub_figure(fig, out_base)
plt.show()
